In [ ]:
import os
import sys

from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())
backend_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(backend_path)
from src.database.database import create_session  # noqa: E402
from src.database import models  # noqa: E402
import nest_asyncio  # noqa: E402

nest_asyncio.apply()

In [ ]:
from sqlalchemy import inspect, func

USER_NAME = "E-stack"
USER_EMAIL = "e-stack@elumini-it.com"
ROLE_NAME = "contributor"

session = create_session()

try:
    User = models.User
    Project = models.Project
    UserProject = models.UserProject
    Role = models.Role


    def get_column_names(model):
        return {column.name for column in inspect(model).columns}


    user_columns = get_column_names(User)

    user = session.query(User).filter(User.email == USER_EMAIL).first()

    if user is None:
        user_kwargs = {
            "email": USER_EMAIL,
            "is_active": True,
            "is_admin": False,
        }

        if "name" in user_columns:
            user_kwargs["name"] = USER_NAME
        elif "username" in user_columns:
            user_kwargs["username"] = USER_NAME

        user = User(**user_kwargs)
        session.add(user)
        session.commit()
        session.refresh(user)
        print(f"User '{USER_NAME}' created with email '{USER_EMAIL}'.")
    else:
        print(f"User with email '{USER_EMAIL}' already exists.")

    collaborator_role = (
        session.query(Role)
        .filter(Role.name == ROLE_NAME)
        .first()
    )

    if collaborator_role is None:
        raise ValueError(
            f"Role '{ROLE_NAME}' not found. "
        )

    projects = session.query(Project).all()
    added_count = 0
    skipped_count = 0

    for project in projects:
        existing_user_project = (
            session.query(UserProject)
            .filter(
                UserProject.user_id == user.id,
                UserProject.project_id == project.id,
            )
            .first()
        )

        if existing_user_project is None:
            session.add(
                UserProject(
                    user_id=user.id,
                    project_id=project.id,
                    role_id=collaborator_role.id,
                )
            )
            added_count += 1
        else:
            if existing_user_project.role_id != collaborator_role.id:
                existing_user_project.role_id = collaborator_role.id
            skipped_count += 1

    session.commit()

    print(
        f"User '{USER_EMAIL}' added as '{ROLE_NAME}' "
        f"to {added_count} projects. Skipped {skipped_count} existing project links."
    )

except Exception:
    session.rollback()
    raise

finally:
    session.close()
